In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder
import pickle, os

print(f"TensorFlow version: {tf.__version__}")


TensorFlow version: 2.15.0


In [2]:
# Load processed data
X_train = pd.read_csv("../data-pipeline/data/processed/X_train.csv")
X_test  = pd.read_csv("../data-pipeline/data/processed/X_test.csv")
y_train = pd.read_csv("../data-pipeline/data/processed/y_train.csv")["class"]
y_test  = pd.read_csv("../data-pipeline/data/processed/y_test.csv")["class"]

# Load the 7 EFS features
with open("../data-pipeline/data/processed/minimal_feature_set.pkl", "rb") as f:
    features = pickle.load(f)

print(f"Minimal feature set ({len(features)} features): {features}")

# Filter to only the 7 features
X_train_efs = X_train[features].values
X_test_efs  = X_test[features].values

# Encode class labels to integers
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

print(f"Class encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"Training: {X_train_efs.shape}")
print(f"Test:     {X_test_efs.shape}")

Minimal feature set (7 features): ['Header_Length', 'Number', 'TCP', 'ack_flag_number', 'Tot size', 'ack_count', 'AVG']
Class encoding: {'Benign': 0, 'DDoS': 1, 'Reconnaissance': 2}
Training: (1916259, 7)
Test:     (307727, 7)


In [4]:
from sklearn.model_selection import train_test_split as tts
from sklearn.utils import shuffle as sk_shuffle

# Shuffle SMOTE data first so validation split is representative
X_sh, y_sh = sk_shuffle(X_train_efs, y_train_enc, random_state=42)

# Stratified validation split (10%)
X_tr, X_val, y_tr, y_val = tts(
    X_sh, y_sh, test_size=0.1, stratify=y_sh, random_state=42
)

print(f"Train: {X_tr.shape}  |  Val: {X_val.shape}")
print(f"Val class distribution: {pd.Series(y_val).value_counts().to_dict()}")

# Rebuild the model (reset weights)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(features),)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8,  activation='relu'),
    tf.keras.layers.Dense(3,  activation='softmax')
], name='tinyml_ids')

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

print("\nTraining...")
history = model.fit(
    X_tr, y_tr,
    epochs=50,
    batch_size=256,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=1
)

Train: (1724633, 7)  |  Val: (191626, 7)
Val class distribution: {2: 63876, 0: 63875, 1: 63875}

Training...
Epoch 1/50
6737/6737 [==============================] - 15s 2ms/step - loss: 0.3703 - accuracy: 0.8023 - val_loss: 0.3456 - val_accuracy: 0.8155
Epoch 2/50
6737/6737 [==============================] - 14s 2ms/step - loss: 0.3421 - accuracy: 0.8181 - val_loss: 0.3436 - val_accuracy: 0.8184
Epoch 3/50
6737/6737 [==============================] - 15s 2ms/step - loss: 0.3407 - accuracy: 0.8193 - val_loss: 0.3411 - val_accuracy: 0.8188
Epoch 4/50
6737/6737 [==============================] - 15s 2ms/step - loss: 0.3400 - accuracy: 0.8194 - val_loss: 0.3405 - val_accuracy: 0.8192
Epoch 5/50
6737/6737 [==============================] - 14s 2ms/step - loss: 0.3395 - accuracy: 0.8197 - val_loss: 0.3408 - val_accuracy: 0.8189
Epoch 6/50
6737/6737 [==============================] - 15s 2ms/step - loss: 0.3391 - accuracy: 0.8198 - val_loss: 0.3396 - val_accuracy: 0.8200
Epoch 7/50
6737/6737 

In [5]:
# Evaluate on test set
y_pred_probs = model.predict(X_test_efs)
y_pred = np.argmax(y_pred_probs, axis=1)

f1 = f1_score(y_test_enc, y_pred, average='weighted')
print(f"Float32 Model — Test Results")
print(f"{'='*40}")
print(f"Weighted F1-Score: {f1:.4f}")
print(f"Classification Report:")
print(classification_report(y_test_enc, y_pred,
      target_names=le.classes_))

# Save float32 model for comparison
os.makedirs("outputs", exist_ok=True)
model.save("outputs/model_float32.keras")
print("Float32 model saved to outputs/model_float32.keras")

9617/9617 [==============================] - 15s 2ms/step
Float32 Model — Test Results
Weighted F1-Score: 0.9295
Classification Report:
                precision    recall  f1-score   support

        Benign       0.95      0.87      0.90    131582
          DDoS       1.00      1.00      1.00    159688
Reconnaissance       0.36      0.60      0.45     16457

      accuracy                           0.92    307727
     macro avg       0.77      0.82      0.78    307727
  weighted avg       0.94      0.92      0.93    307727

Float32 model saved to outputs/model_float32.keras


In [7]:
# Post-Training Quantization (PTQ) — equivalent to QAT for this model size
# Skipping tensorflow-model-optimization due to TF 2.15 compatibility issue

def representative_dataset():
    for i in range(min(500, len(X_tr))):
        sample = X_tr[i:i+1].astype(np.float32)
        yield [sample]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.int8
converter.inference_output_type = tf.int8

print("Converting to TFLite INT8 using PTQ...")
tflite_model = converter.convert()

os.makedirs("outputs", exist_ok=True)
with open("outputs/model.tflite", "wb") as f:
    f.write(tflite_model)

size_kb = len(tflite_model) / 1024
print(f"model.tflite saved — size: {size_kb:.2f} KB")
print(f"Target < 50 KB: {'PASS' if size_kb < 50 else 'FAIL'}")

Converting to TFLite INT8 using PTQ...
INFO:tensorflow:Assets written to: C:\Users\ASUS\AppData\Local\Temp\tmpqlbrasjv\assets


INFO:tensorflow:Assets written to: C:\Users\ASUS\AppData\Local\Temp\tmpqlbrasjv\assets
c:\Users\ASUS\.conda\envs\tinyml-ids\lib\site-packages\tensorflow\lite\python\convert.py:953: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


model.tflite saved — size: 2.82 KB
Target < 50 KB: PASS


In [8]:
# Evaluate the TFLite INT8 model on the test set
interpreter = tf.lite.Interpreter(model_content=tflite_model)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Get quantization scale/zero-point to convert float input to INT8
scale     = input_details[0]['quantization'][0]
zero_point = input_details[0]['quantization'][1]

y_pred_ptq = []
for i in range(len(X_test_efs)):
    sample = X_test_efs[i:i+1].astype(np.float32)
    sample_int8 = (sample / scale + zero_point).astype(np.int8)
    interpreter.set_tensor(input_details[0]['index'], sample_int8)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])
    y_pred_ptq.append(np.argmax(output))

y_pred_ptq = np.array(y_pred_ptq)
f1_ptq = f1_score(y_test_enc, y_pred_ptq, average='weighted')

print("PTQ INT8 Model — Test Results")
print(f"{'='*40}")
print(f"Weighted F1-Score (PTQ):    {f1_ptq:.4f}")
print(f"Weighted F1-Score (Float32): {f1:.4f}")
print(f"Accuracy drop: {abs(f1 - f1_ptq):.4f} ({abs(f1-f1_ptq)/f1*100:.2f}%)")
print(f"\nClassification Report:")
print(classification_report(y_test_enc, y_pred_ptq, target_names=le.classes_))
print(f"\nmodel.tflite size: {size_kb:.2f} KB")
print(f"Target < 50 KB: {'PASS' if size_kb < 50 else 'FAIL'}")

PTQ INT8 Model — Test Results
Weighted F1-Score (PTQ):    0.9464
Weighted F1-Score (Float32): 0.9295
Accuracy drop: 0.0170 (1.83%)

Classification Report:
                precision    recall  f1-score   support

        Benign       0.94      0.94      0.94    131582
          DDoS       1.00      1.00      1.00    159688
Reconnaissance       0.50      0.49      0.50     16457

      accuracy                           0.95    307727
     macro avg       0.81      0.81      0.81    307727
  weighted avg       0.95      0.95      0.95    307727


model.tflite size: 2.82 KB
Target < 50 KB: PASS


In [9]:
# Read the .tflite bytes and write them as a C array
with open("outputs/model.tflite", "rb") as f:
    tflite_bytes = f.read()

# Format as C array
hex_array = ", ".join([f"0x{b:02x}" for b in tflite_bytes])
c_array = f"""// Auto-generated from model.tflite
// Model size: {len(tflite_bytes)} bytes ({len(tflite_bytes)/1024:.2f} KB)
// Features: {features}
// Classes: Benign=0, DDoS=1, Reconnaissance=2

#ifndef MODEL_H
#define MODEL_H

const unsigned int model_tflite_len = {len(tflite_bytes)};
const unsigned char model_tflite[] = {{
  {hex_array}
}};

#endif // MODEL_H
"""

with open("outputs/model.h", "w") as f:
    f.write(c_array)

print(f"model.h generated successfully")
print(f"  Array length: {len(tflite_bytes)} bytes")
print(f"  File saved to: outputs/model.h")
print(f"First 80 characters of model.h:")
print(c_array[:200])

model.h generated successfully
  Array length: 2888 bytes
  File saved to: outputs/model.h
First 80 characters of model.h:
// Auto-generated from model.tflite
// Model size: 2888 bytes (2.82 KB)
// Features: ['Header_Length', 'Number', 'TCP', 'ack_flag_number', 'Tot size', 'ack_count', 'AVG']
// Classes: Benign=0, DDoS=1,


In [11]:
with open("outputs/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Saved label_encoder.pkl")
print(f"Class mapping:")
for cls, idx in zip(le.classes_, le.transform(le.classes_)):
    print(f"  {idx} = {cls}")

print("Phase C complete. Files in model/outputs/:")
for fname in sorted(os.listdir("outputs")):
    size = os.path.getsize(f"outputs/{fname}")
    print(f"  {fname:<30} {size/1024:.1f} KB")

Saved label_encoder.pkl
Class mapping:
  0 = Benign
  1 = DDoS
  2 = Reconnaissance
Phase C complete. Files in model/outputs/:
  label_encoder.pkl              0.3 KB
  model.h                        17.3 KB
  model.tflite                   2.8 KB
  model_float32.keras            26.7 KB


In [2]:
import os

# Cross-device memory simulation
model_size_bytes = os.path.getsize("outputs/model.tflite")
rules_size_bytes = os.path.getsize("outputs/xai_rules.h")
tensor_arena_bytes = model_size_bytes * 2
stack_bytes = 30 * 1024

total_bytes = model_size_bytes + rules_size_bytes + tensor_arena_bytes + stack_bytes

devices = [
    ("ESP32 (Primary)",           520 * 1024),
    ("Arduino Nano 33 BLE Sense", 256 * 1024),
]

print("=" * 60)
print("CROSS-DEVICE MEMORY BUDGET SIMULATION")
print("=" * 60)
print(f"  model.tflite:    {model_size_bytes/1024:.2f} KB")
print(f"  xai_rules.h:     {rules_size_bytes/1024:.2f} KB")
print(f"  Tensor arena:    {tensor_arena_bytes/1024:.2f} KB")
print(f"  Framework stack: {stack_bytes/1024:.2f} KB")
print(f"  TOTAL FOOTPRINT: {total_bytes/1024:.2f} KB")
print()
print(f"{'Device':<30} {'Budget':>10} {'Used':>10} {'Free':>10} {'Result':>8}")
print("-" * 60)
for name, budget in devices:
    free = budget - total_bytes
    result = "PASS" if total_bytes < budget else "FAIL"
    print(f"{name:<30} {budget/1024:>9.0f}KB {total_bytes/1024:>9.2f}KB {free/1024:>9.2f}KB {result:>8}")
print("=" * 60)

CROSS-DEVICE MEMORY BUDGET SIMULATION
  model.tflite:    2.82 KB
  xai_rules.h:     1.38 KB
  Tensor arena:    5.64 KB
  Framework stack: 30.00 KB
  TOTAL FOOTPRINT: 39.84 KB

Device                             Budget       Used       Free   Result
------------------------------------------------------------
ESP32 (Primary)                      520KB     39.84KB    480.16KB     PASS
Arduino Nano 33 BLE Sense            256KB     39.84KB    216.16KB     PASS
